# Stage 4 — signed spatial marker and crossing mechanism

A fixed disorder realization, using the same `L=30`, central `10×10` defect region, central `15×15` integration region, three hexagons and $t_2=0.02$ as in the Stage-4 reports. The notebook computes:

- Signed local-marker maps and mutually exclusive region sums.
- Zero-energy secular roots and exact occupied/unoccupied eigenpairs.
- One-sided finite marker jumps and a **single occupied-state exchange** diagnostic.
- Optional two-/four-state fixed-reference reconstruction (no fitting, frozen deeper occupied background).

**Important:** a marker sum is not a local orbital magnetization; the OBC full-sample marker sums to zero. The finite jump is not proof of a divergent bulk response. The four-state model's accuracy is realization-dependent.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd().resolve()
if not (REPO / 'src').exists():
    REPO = REPO.parent
if not (REPO / 'src').exists():
    raise FileNotFoundError('Start Jupyter in the repository root or notebooks/ directory.')
sys.path.insert(0, str(REPO / 'src'))
import numpy as np
import matplotlib.pyplot as plt
from stage4_local_marker import HaldaneOBC, FigureRegions, exact_marker
from stage4_local_marker.analysis import (secular_roots, root_in_window,
    site_masks, region_sums, exact_crossing_sample,
    occupation_swap_jump, frozen_reference_reconstruction)


In [ ]:
# Full reference setup. Changing L can be expensive (dense N=2 L^2).
L = 30
SEED = 20261008
EPS = 1e-6
model = HaldaneOBC(L,L,t=1.0,t2=0.02,m=0.0)
geo = FigureRegions(model, defect_width=10, integration_width=15)
defects = geo.choose_defects(3, SEED)
H0 = model.pristine()
W = model.defect_unit_operator(defects)
pos = model.positions()
mask = geo.integration_mask()
masks = site_masks(geo, edge_layers=2)
base = exact_marker(H0, pos, max_dim=2*L*L).marker
roots = secular_roots(H0, W)
branch_roots = {'left': root_in_window(roots,-1.5,-.95),
                'right':root_in_window(roots,.6,1.0)}
print('Defects:',defects, 'Roots:',branch_roots)

In [ ]:
samples={}; diagnostics={}
for branch, hc in branch_roots.items():
    minus = exact_crossing_sample(H0,W,pos,mask,base,hc-EPS,max_dim=2*L*L)
    plus  = exact_crossing_sample(H0,W,pos,mask,base,hc+EPS,max_dim=2*L*L)
    samples[branch]=(minus,plus)
    diagnostics[branch]=occupation_swap_jump(minus,plus,pos,base,mask)
    print(f'\n{branch}: h_c={hc:.9f}')
    print('M_R^- / M_R^+:',minus.integrated_marker,plus.integrated_marker)
    print('one-occupied-state swap:',diagnostics[branch])
    print('R/B/edge decomposition (upper side):',region_sums(plus.delta_marker,masks))

## Signed spatial anatomy

Two sites per unit cell are summed only for display. Negative and positive contributions must both remain visible.

In [ ]:
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Rectangle
fig,axes=plt.subplots(2,2,figsize=(11,9),constrained_layout=True)
for i,(branch,(minus,plus)) in enumerate(samples.items()):
    for j,s in enumerate([minus,plus]):
        z=geo.group_site_by_cell(s.delta_marker)
        scale=max(float(np.percentile(abs(z),99)),0.1)
        a=axes[i,j]
        im=a.imshow(z,origin='lower',cmap='RdBu_r',interpolation='none',
                    norm=SymLogNorm(linthresh=.03,vmin=-scale,vmax=scale))
        a.scatter([d[0]+.5 for d in defects],[d[1]+.5 for d in defects],
                  marker='x',color='black',s=40)
        for bounds,color,style in [(geo.defect_bounds,'orange','--'),
                                   (geo.integration_bounds,'lime',':')]:
            a0,a1,b0,b1=bounds
            a.add_patch(Rectangle((a0-.5,b0-.5),a1-a0,b1-b0,fill=False,
                                  ec=color,lw=1.5,ls=style))
        a.set_title(f'{branch}, h_c {"-" if j==0 else "+"} {EPS:g}')
        fig.colorbar(im,ax=a,shrink=.7)
plt.show()

## Optional: two- versus four-state recovery

Leave `RUN_RECONSTRUCTION=False` for the quick mechanism check. Changing it to `True` adds repeated full exact diagonalizations. The calculation follows the historical fixed-reference construction and retains the deep occupied projector.

In [ ]:
RUN_RECONSTRUCTION = False
if RUN_RECONSTRUCTION:
    relative_h = np.array([-.008,-.002,-1e-6,+1e-6,+.002,+.008])
    fig,axs=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
    for a,(branch,hc) in zip(axs,branch_roots.items()):
        hs=hc+relative_h
        truth=np.array([exact_crossing_sample(H0,W,pos,mask,base,h,max_dim=2*L*L)
                        .integrated_marker for h in hs])
        approx2=frozen_reference_reconstruction(H0,W,pos,base,mask,hc,hs,
                                               dimension=2,max_dim=2*L*L)
        approx4=frozen_reference_reconstruction(H0,W,pos,base,mask,hc,hs,
                                               dimension=4,max_dim=2*L*L)
        a.plot(relative_h,truth,'k.',label='Exact')
        a.plot(relative_h,approx2,'--',label='2 states + frozen deep')
        a.plot(relative_h,approx4,'-',label='4 states + frozen deep')
        a.set(xlabel=r'$h-h_c$',ylabel=r'$M_R$',title=branch)
        a.legend()
    plt.show()